# Impetus — Energy-Based Reasoning for Open LLMs
## Kaggle Notebook | Baseline Benchmarking

**Strategy:** Math + Logic first (GSM8K, ARC, BBH)

**Pipeline:** Prompt → N candidates → Energy scoring → Best answer

---
### Instructions
1. Settings → **Accelerator → GPU T4 x1**
2. **Internet** must be ON (Settings → Internet)
3. Run all cells
4. Results saved as CSV in `/kaggle/working/results/`
5. Download CSV or submit to GitHub

In [ ]:
import sys, os, subprocess, json, re
from pathlib import Path

# ── Detect GPU and install compatible PyTorch ──
# P100 (Pascal sm_60) needs torch <= 2.0.1
# T4/P100/others detected via nvidia-smi
gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip().lower()
print(f"Detected GPU: {gpu_info}")

if "p100" in gpu_info:
    print("P100 detected - installing PyTorch 2.0.1 (compatible with sm_60)")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch==2.0.1", "torchvision==0.15.2", "--index-url",
        "https://download.pytorch.org/whl/cu118"])
else:
    print("Modern GPU detected - installing latest PyTorch")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch>=2.1.0", "torchvision>=0.16.0", "--index-url",
        "https://download.pytorch.org/whl/cu118"])

# ── Install rest of dependencies ──
DEPS = [
    "transformers>=4.36.0",
    "accelerate>=0.25.0",
    "datasets>=2.16.0",
    "evaluate>=0.4.1",
    "sentencepiece>=0.1.99",
    "scipy",
    "pandas",
    "tqdm",
]
for dep in DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep])

print("Dependencies installed.")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Device: {gpu_name}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA devices: {torch.cuda.device_count()}")
    # P100 (sm_60) doesn't support bfloat16; T4+ does
    DTYPE = torch.float16 if "P100" in gpu_name else torch.bfloat16
    print(f"Using dtype: {DTYPE}")

In [ ]:
# ── Clone project ──
REPO_URL = "https://github.com/EdhieBM/Impetus.git"
PROJECT_DIR = "/kaggle/working/Impetus"

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, PROJECT_DIR])

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working in: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded successfully")

In [ ]:
# ── Phase 1: Baseline benchmark (GSM8K) ──
from benchmarks.run_baseline import run_gsm8k, save_results

print("Running GSM8K baseline (50 samples for speed)...")
results = run_gsm8k(model, tokenizer, max_samples=50)
path = save_results(results, MODEL_NAME, "results")
print(f"Saved: {path}")

In [ ]:
# ── Evaluate ──
from benchmarks.evaluate_results import evaluate_gsm8k

stats = evaluate_gsm8k(results)
print(f"GSM8K  | Acc: {stats['accuracy']:.2%}  ({stats['correct']}/{stats['total']})  | Latency: {stats['avg_latency_s']:.2f}s")

In [ ]:
# ── Phase 2: KONA verifier quick test ──
from reranker.kona_verifier import KONAVerifier

verifier = KONAVerifier(MODEL_NAME)
prompt = "Solve: A farmer has 12 apples. He gives 3 to his neighbor and eats 2. How many are left?"
best, score, candidates = verifier.rerank(prompt, n=3)

print(f"Prompt: {prompt}\n")
print(f"Best answer (energy={score:.4f}): {best}\n")
print("All candidates:")
for i, (ans, s) in enumerate(candidates):
    print(f"  [{i}] energy={s:.4f}: {ans}")

In [ ]:
import pandas as pd
print("\n=== Session Summary ===")
print(f"Model: {MODEL_NAME}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"Baseline results: {stats}")

pd.DataFrame(results).to_csv("results/kaggle_baseline_full.csv", index=False)
print("\nCSV saved to results/kaggle_baseline_full.csv")